In [1]:
from pyspark.sql import SparkSession
import logging

logging.getLogger("py4j").setLevel(logging.ERROR)

In [2]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0"
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "spark-warehouse-iceberg")
    
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
26/04/28 20:43:58 WARN Utils: Your hostname, L-1-10-43-28 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 20:43:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/gustavo/trabalho-eng-dados-arquitetura-dados/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gustavo/.ivy2/cache
The jars for the packages stored in: /home/gustavo/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0b0419c8-4f1d-4fc4-8b47-d48b0b3bf56f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.0/iceberg-spark-runtime-3.5_2.12-1.5.0.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0!iceberg-spark-runtime-3.5_2.12.jar (11029ms)
:: resolution report :: resolve 556ms :: artifacts dl 11031ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evi

In [3]:
spark

In [5]:
spark.sql("""
    CREATE TABLE local.db.ar_condicionado_iceberg (
        id INT,
        nm_modelo STRING,
        nm_marca STRING,
        valor DECIMAL(10,2),
        ano INT,
        potencia STRING
    )
    USING iceberg
""")

DataFrame[]

In [6]:
spark.sql("SELECT * FROM local.db.ar_condicionado_iceberg").show()

+---+---------+--------+-----+---+--------+
| id|nm_modelo|nm_marca|valor|ano|potencia|
+---+---------+--------+-----+---+--------+
+---+---------+--------+-----+---+--------+



In [7]:
spark.sql(
    """
    SELECT * 
    FROM local.db.ar_condicionado_iceberg.snapshots
    """
).show(truncate=False)

+------------+-----------+---------+---------+-------------+-------+
|committed_at|snapshot_id|parent_id|operation|manifest_list|summary|
+------------+-----------+---------+---------+-------------+-------+
+------------+-----------+---------+---------+-------------+-------+



In [10]:
spark.sql("""
    INSERT INTO local.db.ar_condicionado_iceberg 
    (id, nm_modelo, nm_marca, valor, ano, potencia)
    VALUES 
    (1, 'KOHI 09QC 1HV', 'KOMECO', 1798.90, 2023, '9000 BTU/h'),
    (2, 'B0DSLQP69S', 'SAMSUNG', 2089.10, 2024, '12000 BTUs'),
    (3, '42AGVCC30M5', 'SAMSUNG', 6590.00, 2025, '30000 BTUs')
""")

DataFrame[]

In [12]:
spark.sql("select * from local.db.ar_condicionado_iceberg").show()

+---+-------------+--------+-------+----+----------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|
+---+-------------+--------+-------+----+----------+
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|
+---+-------------+--------+-------+----+----------+



In [13]:
spark.sql(
    """
    ALTER TABLE local.db.ar_condicionado_iceberg 
      ADD COLUMN fl_inverter BOOLEAN
    """
)

spark.sql(
    """
    ALTER TABLE local.db.ar_condicionado_iceberg 
      ADD COLUMN tensao STRING
    """
)

DataFrame[]

In [15]:
spark.sql(
    """
    select * from local.db.ar_condicionado_iceberg
    """
).show()

+---+-------------+--------+-------+----+----------+-----------+------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|fl_inverter|tensao|
+---+-------------+--------+-------+----+----------+-----------+------+
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|       NULL|  NULL|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|       NULL|  NULL|
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|       NULL|  NULL|
+---+-------------+--------+-------+----+----------+-----------+------+



In [16]:
spark.sql(
    """
    UPDATE local.db.ar_condicionado_iceberg
    SET fl_inverter = true, tensao = '220v'
    WHERE id IN (2, 3)
    """
)

DataFrame[]

In [18]:
spark.sql(
    """
    select * from local.db.ar_condicionado_iceberg
    """
).show()

+---+-------------+--------+-------+----+----------+-----------+------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|fl_inverter|tensao|
+---+-------------+--------+-------+----+----------+-----------+------+
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|       NULL|  NULL|
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|       true|  220v|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|       true|  220v|
+---+-------------+--------+-------+----+----------+-----------+------+



In [19]:
spark.sql(
    """
    UPDATE local.db.ar_condicionado_iceberg
    SET fl_inverter = false, tensao = '170v'
    WHERE id = 1
    """
)

DataFrame[]

In [20]:
spark.sql(
    """
    DELETE FROM local.db.ar_condicionado_iceberg WHERE id = 1
    """
    )

DataFrame[]

In [27]:
spark.sql(
    """
    select * from local.db.ar_condicionado_iceberg
    """
).show()

+---+-----------+--------+-------+----+----------+-----------+------+
| id|  nm_modelo|nm_marca|  valor| ano|  potencia|fl_inverter|tensao|
+---+-----------+--------+-------+----+----------+-----------+------+
|  3|42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|       true|  220v|
|  2| B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|       true|  220v|
+---+-----------+--------+-------+----+----------+-----------+------+

